### ETL para gerar .parquet e subir no git 
#### na pratica se for replicar por ai, so salvar o .parquet e ser feliz, caso queira fazer por si proprio, eis o ETL de exemplo: 

In [1]:
import pandas as pd
import glob
import os

# --- 1️⃣ Definir pasta com os CSVs ---
pasta_csv = r"C:\Users\jgeov\Downloads\Help_desk_GLPI_IFPR"
arquivos_csv = glob.glob(os.path.join(pasta_csv, "*.csv"))

# --- 2️⃣ Ler e concatenar todos os CSVs ---
lista_df = []
for arquivo in arquivos_csv:
    df = pd.read_csv(arquivo, encoding="utf-8", sep=";", engine="python")  # engine python tolera quebras de linha
    lista_df.append(df)

df_total = pd.concat(lista_df, ignore_index=True)

# --- 3️⃣ Selecionar colunas importantes (manter todas para referência) ---
colunas_importantes = ['Ticker', 'Requerente', 'Grupo', 'Categoria',
                       'Data Abertura', 'Data Solução', 'Data Fechamento',
                       'Atendente', 'Satisfação', 'Data Iteração',
                       'Usuário Iteração', 'Resolvido']
df_total = df_total[colunas_importantes]

# --- 4️⃣ Criar coluna de texto consolidado para NLP ---
df_total["texto"] = df_total["Ticker"].fillna("").astype(str) + " " + df_total["Requerente"].fillna("").astype(str)

# --- 5️⃣ Filtrar apenas linhas com texto e categoria (Grupo) ---
df_total = df_total[(df_total["texto"] != "") & (df_total["Grupo"].notna())]

# --- 6️⃣ Remover duplicatas (com base em texto e Grupo) ---
df_total = df_total.drop_duplicates(subset=["texto", "Grupo"])

# --- 7️⃣ Salvar em Parquet mantendo todas as colunas ---
saida_parquet = "ifpr_helpdesk_completo.parquet"
df_total.to_parquet(saida_parquet, index=False)

print(f"Base filtrada e salva em Parquet com todas as colunas: {saida_parquet}")

Base filtrada e salva em Parquet com todas as colunas: ifpr_helpdesk_completo.parquet
